In [ ]:
pip install statsmodels
pip install yfinance

  Using cached statsmodels-0.14.6-cp311-cp311-win_amd64.whl.metadata (9.8 kB)
  Using cached patsy-1.0.2-py2.py3-none-any.whl.metadata (3.6 kB)
Using cached statsmodels-0.14.6-cp311-cp311-win_amd64.whl (9.6 MB)
Using cached patsy-1.0.2-py2.py3-none-any.whl (233 kB)

   ---------------------------------------- 0/2 [patsy]
   ---------------------------------------- 0/2 [patsy]
   ---------------------------------------- 0/2 [patsy]
   ---------------------------------------- 0/2 [patsy]
   -------------------- ------------------- 1/2 [statsmodels]
   -------------------- ------------------- 1/2 [statsmodels]
   -------------------- ------------------- 1/2 [statsmodels]
   -------------------- ------------------- 1/2 [statsmodels]
   -------------------- ------------------- 1/2 [statsmodels]
   -------------------- ------------------- 1/2 [statsmodels]
   -------------------- ------------------- 1/2 [statsmodels]
   -------------------- ------------------- 1/2 [statsmodels]
   ----------

In [23]:
"""
Pipeline 01 - S&P500 Historical Risk Measures

Objetivo:
- Download do S&P500 via Yahoo Finance
- Limpeza da série
- Retornos logarítmicos
- Estatísticas descritivas
- Testes Jarque-Bera e ADF
- Rolling Historical VaR
- Rolling Historical Expected Shortfall
- Rolling Volatility
- Base mensal para integração com TRACE
"""

import numpy as np
import pandas as pd
import yfinance as yf

from scipy import stats
from statsmodels.tsa.stattools import adfuller

# ==========================================================
# CONFIGURAÇÕES
# ==========================================================

START_DATE = "2008-01-01"
END_DATE = "2025-12-31"

ROLLING_WINDOW = 252

# ==========================================================
# DOWNLOAD
# ==========================================================

def download_sp500(start=START_DATE,
                   end=END_DATE):

    df = yf.download(
        "^GSPC",
        start=start,
        end=end,
        auto_adjust=True,
        progress=False,
        multi_level_index=False
    )
    df = df.reset_index()

    df = df.rename(columns={
        "Date":"date",
        "Close":"sp500"
    })

    return df[["date","sp500"]]


# ==========================================================
# LIMPEZA
# ==========================================================

def clean_sp500(df):

    df = df.copy()

    df["date"] = pd.to_datetime(df["date"])

    df = (
        df
        .drop_duplicates("date")
        .sort_values("date")
    )

    df["sp500"] = df["sp500"].ffill()

    df = df[df["sp500"] > 0]

    df["log_return"] = np.log(
        df["sp500"] /
        df["sp500"].shift(1)
    )

    df["return"] = df["sp500"].pct_change()

    df = df.dropna().reset_index(drop=True)

    return df


# ==========================================================
# TESTES
# ==========================================================

def diagnostics(df):

    r = df["log_return"]

    jb = stats.jarque_bera(r)

    adf = adfuller(r)

    print("="*60)
    print("ESTATÍSTICAS DESCRITIVAS")
    print("="*60)

    print(r.describe())

    print("\nJarque-Bera")
    print(jb)

    print("\nADF")
    print(f"Statistic : {adf[0]:.4f}")
    print(f"P-value   : {adf[1]:.6f}")


# ==========================================================
# EXPECTED SHORTFALL
# ==========================================================

def expected_shortfall(x, alpha):

    x = np.asarray(x)

    var = np.quantile(x, alpha)

    return x[x <= var].mean()


# ==========================================================
# HISTORICAL RISK
# ==========================================================

def rolling_risk(df,
                 window=252):

    df = df.copy()

    roll = df["log_return"].rolling(window)

    df["rolling_volatility"] = (
        roll.std() *
        np.sqrt(252)
    )

    df["rolling_skewness"] = roll.skew()

    df["rolling_kurtosis"] = roll.kurt()

    df["hist_var95"] = roll.quantile(0.05)

    df["hist_var99"] = roll.quantile(0.01)

    df["hist_es95"] = roll.apply(
        lambda x: expected_shortfall(x,0.05),
        raw=True
    )

    df["hist_es99"] = roll.apply(
        lambda x: expected_shortfall(x,0.01),
        raw=True
    )

    return df


# ==========================================================
# BASE MENSAL
# ==========================================================

def monthly_dataset(df):

    temp = df.copy()

    temp = temp.set_index("date")

    monthly = temp.resample("ME").agg(

        sp500=("sp500","last"),

        monthly_return=(
            "log_return",
            lambda x: np.exp(x.sum())-1
        ),

        monthly_volatility=(
            "log_return",
            lambda x: x.std()*np.sqrt(252)
        ),

        hist_var95=("hist_var95","last"),

        hist_var99=("hist_var99","last"),

        hist_es95=("hist_es95","last"),

        hist_es99=("hist_es99","last"),

        rolling_volatility=(
            "rolling_volatility",
            "last"
        ),

        rolling_skewness=(
            "rolling_skewness",
            "last"
        ),

        rolling_kurtosis=(
            "rolling_kurtosis",
            "last"
        ),

        trading_days=(
            "log_return",
            "count"
        )

    )

    # NÃO remover observações.
    # Os primeiros meses terão NaN porque ainda não existem
    # 252 observações para cálculo do VaR.

    monthly = monthly.reset_index()

    return monthly


# ==========================================================
# EXECUÇÃO
# ==========================================================

if __name__ == "__main__":

    print("Baixando dados...")

    daily = download_sp500()

    print("\nPeríodo da base:")
    print(daily["date"].min())
    print(daily["date"].max())

    print(f"\nNúmero de observações: {len(daily):,}")

    daily = clean_sp500(daily)

    diagnostics(daily)

    daily = rolling_risk(
        daily,
        window=ROLLING_WINDOW
    )

    monthly = monthly_dataset(daily)

    print("\nBase mensal:")

    print(monthly.head())

    print(monthly.tail())

    # Exportação


    monthly.to_csv(
        "sp500_monthly.csv",
        index=False
    )


    print("\nArquivos exportados com sucesso.")

Baixando dados...

Período da base:
2008-01-02 00:00:00
2025-12-30 00:00:00

Número de observações: 4,528
ESTATÍSTICAS DESCRITIVAS
count    4527.000000
mean        0.000345
std         0.012659
min        -0.127652
25%        -0.004142
50%         0.000721
75%         0.005891
max         0.109572
Name: log_return, dtype: float64

Jarque-Bera
SignificanceResult(statistic=np.float64(30097.854105467643), pvalue=np.float64(0.0))

ADF
Statistic : -15.0552
P-value   : 0.000000

Base mensal:
        date        sp500  monthly_return  monthly_volatility  hist_var95  \
0 2008-01-31  1378.550049       -0.047410            0.245686         NaN   
1 2008-02-29  1330.630005       -0.034761            0.206491         NaN   
2 2008-03-31  1322.699951       -0.005960            0.287655         NaN   
3 2008-04-30  1385.589966        0.047547            0.181756         NaN   
4 2008-05-31  1400.380005        0.010674            0.143770         NaN   

   hist_var99  hist_es95  hist_es99  rolling_v